# Base nacional de fatores — transporte rodoviário de cargas

Pipeline de extração rodando no Google Colab, com os dados persistidos no
Google Drive.

**Regra que não muda:** a sessão do Colab é apagada quando você fecha. Tudo que
precisa sobreviver — arquivos brutos e Parquet extraídos — vive no Drive, não
em `/content`.

## 1. Ambiente

In [ ]:
import sys, subprocess, importlib

for pacote in ["pyarrow", "openpyxl", "pandas"]:
    try:
        importlib.import_module(pacote)
        print(f"{pacote}: ok")
    except ImportError:
        print(f"{pacote}: instalando...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote],
                       check=True)

import pandas as pd
print("pandas", pd.__version__)

## 2. Montar o Drive

O Colab vai pedir autorização. Depois disso, `/content/drive/MyDrive` é a sua
pasta do Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Estrutura do projeto no Drive

Rode uma vez. Cria as pastas e, se o projeto ainda não existir, avisa o que
fazer. Suba o conteúdo do zip (`codigo/`, `crosswalk/`, `README.md`) para
`MyDrive/base-fatores-br/`.

In [ ]:
from pathlib import Path

RAIZ = Path('/content/drive/MyDrive/base-fatores-br')

for sub in ['dados/00_brutos', 'dados/10_extraidos', 'dados/20_harmonizados',
            'dados/30_publicados', 'codigo', 'crosswalk', 'docs']:
    (RAIZ / sub).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(RAIZ / 'codigo'))
%cd {RAIZ}

scripts = sorted(p.name for p in (RAIZ / 'codigo').glob('*.py'))
print('raiz:', RAIZ)
print('scripts encontrados:', scripts or 'NENHUM — suba a pasta codigo/ para o Drive')

## 4. Registrar um arquivo bruto

Toda fonte baixada passa por aqui. A função calcula o SHA-256, copia para
`dados/00_brutos/` e grava a linha no `MANIFESTO.csv`.

Isso não é burocracia: é o que permite a outra pessoa confirmar que baixou
exatamente o mesmo arquivo que você usou. Sem o hash, a base não é reprodutível
— é só um CSV com um DOI em cima.

In [ ]:
import hashlib, shutil
from datetime import date

MANIFESTO = RAIZ / 'dados/00_brutos/MANIFESTO.csv'
COLUNAS = ['id_fonte', 'arquivo', 'instituicao', 'descricao', 'url_origem',
           'ano_ref', 'data_acesso', 'sha256', 'formato', 'licenca', 'observacao']


def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 20), b''):
            h.update(bloco)
    return h.hexdigest()


def registra(origem, id_fonte, instituicao, url_origem, ano_ref,
             descricao='', licenca='', observacao=''):
    origem = Path(origem)
    if not origem.exists():
        raise FileNotFoundError(origem)

    destino = RAIZ / 'dados/00_brutos' / origem.name
    if origem.resolve() != destino.resolve():
        shutil.copy2(origem, destino)

    h = sha256(destino)
    linha = {
        'id_fonte': id_fonte, 'arquivo': destino.name, 'instituicao': instituicao,
        'descricao': descricao, 'url_origem': url_origem, 'ano_ref': ano_ref,
        'data_acesso': date.today().isoformat(), 'sha256': h,
        'formato': destino.suffix.lstrip('.'), 'licenca': licenca,
        'observacao': observacao,
    }

    if MANIFESTO.exists():
        man = pd.read_csv(MANIFESTO, dtype=str)
        for c in COLUNAS:
            if c not in man.columns:
                man[c] = pd.NA
        # placeholder da mesma fonte sem arquivo preenchido: substitui
        vazio = (man['id_fonte'] == id_fonte) & man['arquivo'].isna()
        man = man[~vazio]
        man = man[~((man['id_fonte'] == id_fonte) & (man['arquivo'] == destino.name))]
        man = pd.concat([man, pd.DataFrame([linha])], ignore_index=True)
    else:
        man = pd.DataFrame([linha])

    man[COLUNAS].to_csv(MANIFESTO, index=False)
    print(f"registrado: {destino.name}\n  sha256: {h[:16]}...  ({len(man)} linhas no manifesto)")
    return destino

### Subir o arquivo

Duas opções. Pelo painel lateral do Colab (ícone de pasta) arrastando para
`/content`, ou pelo seletor abaixo.

In [ ]:
from google.colab import files

enviados = files.upload()          # abre o seletor
nome = next(iter(enviados))        # primeiro arquivo enviado
print('recebido:', nome)

In [ ]:
# Ajuste os metadados antes de rodar.
caminho = registra(
    origem      = nome,
    id_fonte    = 'anp_vendas_diesel',
    instituicao = 'ANP',
    url_origem  = 'COLE A URL EXATA DE ONDE BAIXOU',
    ano_ref     = 2024,
    descricao   = 'Vendas de diesel por UF e produto',
    licenca     = 'Dados abertos',
)

## 5. Extrair

Os scripts rodam por linha de comando, igual fora do Colab. O `!` executa no
shell.

In [ ]:
!python codigo/02_extrai_anp.py "{caminho}" --ano 2024

Para a RNTRC, **inspecione antes**. O esquema da ANTT não é estável e o
script não adivinha nomes de coluna.

In [ ]:
# !python codigo/03_extrai_rntrc.py "{caminho}" --inspecionar

In [ ]:
# Depois de salvar o mapa sugerido em crosswalk/rntrc_colunas.csv:
# !python codigo/03_extrai_rntrc.py "{caminho}" --ano 2024

## 6. Conferir o resultado

In [ ]:
extraidos = sorted((RAIZ / 'dados/10_extraidos').glob('*.parquet'))
for p in extraidos:
    print(f"{p.name}  ({p.stat().st_size/1024:.0f} KB)")

if extraidos:
    df = pd.read_parquet(extraidos[-1])
    print(f"\n{len(df):,} linhas  |  colunas: {list(df.columns)}")
    display(df.head(10))

## 7. Versionamento — o ponto fraco do Colab

O Drive guarda os arquivos, mas **não é controle de versão**. Ele não registra
por que uma linha do crosswalk mudou, nem permite voltar a um estado anterior
de forma confiável.

Para uma base que será citada, o código e o crosswalk precisam estar no Git. O
caminho que funciona bem com Colab:

- `codigo/` e `crosswalk/` vivem no **GitHub** (é o que será citado e auditado)
- `dados/00_brutos/` e `dados/10_extraidos/` vivem no **Drive** (arquivos
  grandes, não versionáveis)

A célula abaixo clona o repositório para dentro da sessão e aponta os imports
para lá. Preencha a URL quando o repositório existir.

In [ ]:
# REPO = 'https://github.com/<usuario>/base-fatores-transporte-br.git'
# !git clone {REPO} /content/repo
# sys.path.insert(0, '/content/repo/codigo')
# %cd /content/repo
# Os dados continuam sendo lidos e escritos em RAIZ (Drive).